In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Higienização

In [2]:
import pandas as pd

# Carregar o dataset
# Certifique-se de que o arquivo 'earthquakes.csv' fez o upload no Colab
df = pd.read_csv('Dataset/earthquakes.csv')

# Exibir as primeiras linhas originais e o tamanho do dataset
print(f"Tamanho original do dataset: {df.shape}")
display(df.head())

Tamanho original do dataset: (1137, 43)


,id,magnitude,type,title,date,time,updated,url,detailUrl,felt,...,location,continent,country,subnational,city,locality,postcode,what3words,timezone,locationDetails
0,us7000necw,4.8,earthquake,"M 4.8 - 33 km WSW of Ackerly, Texas",2024-09-17T00:49:42,1726534182289,1726583895255,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,1893,...,"Ackerly, Texas",North America,United States of America (the),Texas,Tarzan-Lenorah,Tarzan-Lenorah,79783.0,landmass.perkily.affords,-300,"[{'id': '80684', 'wikidataId': '', 'name': '79..."
1,tx2024shcj,5.1,earthquake,"M 5.1 - 34 km WSW of Ackerly, Texas",2024-09-17T00:49:42,1726534182183,1726672002991,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,2042,...,"Ackerly, Texas",North America,United States of America (the),Texas,Tarzan-Lenorah,Tarzan-Lenorah,79331.0,escalator.grownups.dwell,-300,"[{'id': '89341', 'wikidataId': '', 'name': '48..."
2,ci40734823,3.7,earthquake,"M 3.7 - 6 km N of Malibu, CA",2024-09-16T11:22:08,1726485728190,1726637414586,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,1580,...,"Malibu, CA",North America,United States of America (the),California,Los Angeles,Agoura Hills-Malibu,90265.0,clocking.uploaded.issuer,-420,"[{'id': '93478', 'wikidataId': 'Q844837', 'nam..."
3,tx2024scvz,3.9,earthquake,"M 3.9 - 58 km S of Whites City, New Mexico",2024-09-14T17:01:06,1726333266539,1726584426218,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,5,...,"Whites City, New Mexico",North America,United States of America (the),Texas,Van Horn,Van Horn,NaN,sailboats.sawn.speeding,-300,"[{'id': '9', 'wikidataId': 'Q49', 'name': 'Nor..."
4,us7000ndte,4.1,earthquake,"M 4.1 - 60 km S of Whites City, New Mexico",2024-09-14T17:01:06,1726333266382,1726334616179,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,4,...,"Whites City, New Mexico",North America,United States of America (the),Texas,Van Horn,Van Horn,NaN,spinners.downtime.computes,-300,"[{'id': '9', 'wikidataId': 'Q49', 'name': 'Nor..."


In [3]:
# Lista de colunas para remover baseada na análise prévia
colunas_para_remover = [
    'id', 'code', 'url', 'detailUrl', 'what3words', 'ids', # Identificadores e Links
    'type', 'geometryType',                                # Valores constantes
    'updated', 'time',                                     # Metadados de sistema/timestamps duplicados
    'title', 'locationDetails', 'place', 'placeOnly',      # Textos redundantes ou complexos
    'location', 'city', 'locality',                        # Textos geográficos propensos a erros
    'postcode'                                             # Excesso de dados nulos
]

# Removendo as colunas
df_clean = df.drop(columns=colunas_para_remover)

print(f"Tamanho após remoção de colunas: {df_clean.shape}")

Tamanho após remoção de colunas: (1137, 25)


In [4]:
# Verificando a quantidade de nulos antes do tratamento
print("Valores nulos antes do tratamento:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# Preenchendo valores nulos em colunas categóricas com 'Unknown'
colunas_categoricas_com_nulos = ['alert', 'continent', 'country', 'subnational']
for col in colunas_categoricas_com_nulos:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('Unknown')

# Se houver alguma outra linha que ainda tenha nulos (dados corrompidos), nós a removemos
df_clean = df_clean.dropna()

print("\nValores nulos após o tratamento:")
print(df_clean.isnull().sum().max()) # Deve retornar 0
print(f"Tamanho após tratamento de nulos: {df_clean.shape}")

Valores nulos antes do tratamento:
alert          373
continent      270
country        338
subnational    421
dtype: int64

Valores nulos após o tratamento:
0
Tamanho após tratamento de nulos: (1137, 25)


In [5]:
# Converter a coluna 'date' para o formato datetime
df_clean['date'] = pd.to_datetime(df_clean['date'])

# (Opcional) Extrair ano, mês e hora para ajudar modelos de Machine Learning
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['day'] = df_clean['date'].dt.day
#df_clean['hour'] = df_clean['date'].dt.hour

# Exibir os tipos de dados atualizados
print(df_clean.dtypes)

magnitude             float64
date           datetime64[us]
felt                    int64
cdi                     int64
mmi                     int64
alert                     str
status                    str
tsunami                 int64
sig                     int64
net                       str
sources                   str
types                     str
nst                     int64
dmin                  float64
rms                   float64
gap                   float64
magType                   str
depth                 float64
latitude              float64
longitude             float64
distanceKM              int64
continent                 str
country                   str
subnational               str
timezone                int64
year                    int32
month                   int32
day                     int32
dtype: object


### Geopy

In [6]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import time

In [7]:
from geopy.geocoders import Nominatim
import time

# Inicializar a API
geolocator = Nominatim(user_agent="meu_projeto_terremotos_ia")

# 1. Identificar quais linhas estão com o país "Unknown" (ou nulo)
# Assumindo que você usou o código anterior e preencheu os nulos com 'Unknown'
filtro_sem_pais = df_clean['country'] == 'Unknown'

# Mostrar quantos países faltam
quantidade_faltante = filtro_sem_pais.sum()
print(f"Buscando o país exato para {quantidade_faltante} registros...")

# 2. Criar a função otimizada para a API
def atualizar_pais_faltante(linha):
    coord = f"{linha['latitude']}, {linha['longitude']}"
    try:
        time.sleep(1) # Pausa obrigatória de 1 segundo
        loc = geolocator.reverse(coord, language='en', timeout=10)

        # Se encontrou um local com endereço e país
        if loc and 'address' in loc.raw and 'country' in loc.raw['address']:
            return loc.raw['address']['country']

        # Se a API funcionou, mas não retornou país, é porque está no oceano!
        return 'Ocean/Offshore'

    except Exception as e:
        # Se der erro na conexão, mantém como Unknown para tentarmos depois
        return 'Unknown'

# 3. Aplicar a função APENAS nas linhas filtradas
# Usamos o .loc para atualizar o df_clean original diretamente na coluna 'country'
if quantidade_faltante > 0:
    df_clean.loc[filtro_sem_pais, 'country'] = df_clean[filtro_sem_pais].apply(atualizar_pais_faltante, axis=1)

print("\nBusca concluída! Veja como ficou a distribuição dos países agora:")
# 4. Verificar o resultado final (quais países a IA agora conhece)
print(df_clean['country'].value_counts().head(10))

Buscando o país exato para 338 registros...

Busca concluída! Veja como ficou a distribuição dos países agora:
country
United States of America (the)    423
Ocean/Offshore                    335
Japan                              68
Taiwan (Province of China)         55
Indonesia                          38
China                              30
Chile                              29
Afghanistan                        26
Argentina                          26
Philippines (the)                  17
Name: count, dtype: int64


In [8]:
import pandas as pd

# 1. Obter as contagens do DataFrame original (df) e do limpo (df_clean)
# Usamos dropna=False no df original para ele contar os nulos também!
contagem_antes = df['country'].value_counts(dropna=False)
contagem_depois = df_clean['country'].value_counts()

# 2. Criar um novo DataFrame que junta as duas contagens lado a lado
df_comparacao = pd.DataFrame({
    'Antes': contagem_antes,
    'Depois': contagem_depois
})

# 3. Tratar os buracos (países que existem em um mas não no outro)
# fillna(0) troca os vazios por zero, e astype(int) deixa os números inteiros
df_comparacao = df_comparacao.fillna(0).astype(int)

# 4. Criar a coluna da sua lógica matemática (A Diferença)
df_comparacao['Diferença'] = df_comparacao['Depois'] - df_comparacao['Antes']

# 5. Exibir o resultado, ordenando pelas categorias que mais mudaram
display(df_comparacao.sort_values(by='Diferença', ascending=False))

,Antes,Depois,Diferença
country,,,
Ocean/Offshore,0,335,335
Greece,0,3,3
Argentina,26,26,0
Afghanistan,26,26,0
Brazil,2,2,0
Bangladesh,2,2,0
Chile,29,29,0
China,30,30,0
Congo (the Democratic Republic of the),1,1,0


In [9]:
# 1. Verificar quantas linhas estão exatamente duplicadas
quantidade_duplicados = df_clean.duplicated().sum()
print(f"Linhas duplicadas encontradas: {quantidade_duplicados}")

df_clean = df_clean.drop_duplicates()

Linhas duplicadas encontradas: 535


In [10]:
quantidade_duplicados = df_clean.duplicated().sum()
print(f"Linhas duplicadas encontradas: {quantidade_duplicados}")

Linhas duplicadas encontradas: 0


In [11]:
# Visualizar o dataset higienizado
display(df_clean.head())

# Salvar o novo dataset limpo
arquivo_limpo = 'Dataset/earthquakes_cleaned.csv'
df_clean.to_csv(arquivo_limpo, index=False)

print(f"\nHigienização concluída! O dataset limpo foi salvo como: {arquivo_limpo}")

,magnitude,date,felt,cdi,mmi,alert,status,tsunami,sig,net,...,latitude,longitude,distanceKM,continent,country,subnational,timezone,year,month,day
0,4.8,2024-09-17 00:49:42,1893,6,5,green,reviewed,0,994,us,...,32.3984,-102.044,33,North America,United States of America (the),Texas,-300,2024,9,17
1,5.1,2024-09-17 00:49:42,2042,6,5,green,reviewed,0,1040,tx,...,32.4140,-102.057,34,North America,United States of America (the),Texas,-300,2024,9,17
2,3.7,2024-09-16 11:22:08,1580,4,4,Unknown,reviewed,0,591,ci,...,34.0678,-118.807,6,North America,United States of America (the),California,-420,2024,9,16
3,3.9,2024-09-14 17:01:06,5,3,4,green,reviewed,0,236,tx,...,31.6470,-104.450,58,North America,United States of America (the),Texas,-300,2024,9,14
4,4.1,2024-09-14 17:01:06,4,3,4,green,reviewed,0,260,us,...,31.6323,-104.473,60,North America,United States of America (the),Texas,-300,2024,9,14



Higienização concluída! O dataset limpo foi salvo como: Dataset/earthquakes_cleaned.csv


In [13]:
print(df.isnull().sum())

id                   0
magnitude            0
type                 0
title                0
date                 0
time                 0
updated              0
url                  0
detailUrl            0
felt                 0
cdi                  0
mmi                  0
alert              373
status               0
tsunami              0
sig                  0
net                  0
code                 0
ids                  0
sources              0
types                0
nst                  0
dmin                 0
rms                  0
gap                  0
magType              0
geometryType         0
depth                0
latitude             0
longitude            0
place                0
distanceKM           0
placeOnly            0
location             0
continent          270
country            338
subnational        421
city               463
locality             0
postcode           940
what3words           0
timezone             0
locationDetails      0
dtype: int6

In [14]:
print(df_clean.isnull().sum())

magnitude      0
date           0
felt           0
cdi            0
mmi            0
alert          0
status         0
tsunami        0
sig            0
net            0
sources        0
types          0
nst            0
dmin           0
rms            0
gap            0
magType        0
depth          0
latitude       0
longitude      0
distanceKM     0
continent      0
country        0
subnational    0
timezone       0
year           0
month          0
day            0
dtype: int64
